In [1]:
import numpy as np
from os import listdir
from os.path import (
    join as join_path,
    isdir
)

from json import load as load_json

In [2]:
def load_json_dict(path):
    try :
        with open(path) as f:
            d = load_json(f)
    except FileNotFoundError:
        return None
    return d
def update_results_dict(comp_name, dname, expno, results, eval_tracker):
    if not comp_name in results:
        results[comp_name] = {}
    if not dname in results[comp_name]:
        results[comp_name][dname] = {}
    if not expno in results[comp_name][dname]:
        results[comp_name][dname][expno] = {}
    results[comp_name][dname][expno] = eval_tracker
    return results
def summarize_results(huge_dict):
    # Structure: {dataset: {loss_name: {labeling_method: {evaluation_metric: "mean±std"}}}}
    summary = {}

    for algorithm, datasets in huge_dict.items():
        for dataset_name, experiments in datasets.items():
            if dataset_name not in summary:
                summary[dataset_name] = {}

            if algorithm not in summary[dataset_name]:
                summary[dataset_name][algorithm] = {}

            # Collect values per metric
            metrics_values = {}

            for exp_num, evals in experiments.items():
                if evals is None:
                    continue
                for metric, values in evals.items():
                    if values is not None and isinstance(values, list) and values:
                        metrics_values.setdefault(metric, []).append(values[-1])

            # Calculate mean and std
            for metric, values in metrics_values.items():
                if isinstance(values, list):
                    if metric == "no_labeled_pts":
                        continue
                    mean = np.mean(values)
                    std = np.std(values)
                    summary_str = f"{mean:.2f}±{std:.2f}"
                else:
                    summary_str = "-"
                summary[dataset_name][algorithm][metric] = summary_str

            # Handle case when evals is None or no metrics at all
            all_metrics = set(metrics_values.keys())
            if not all_metrics and experiments:
                summary[dataset_name][algorithm] = "-"
                    
    return summary


In [8]:
experiment_namecode = "competitors203"
experiment_path = f"/export/share/peters57dm/Verbund/deepsync/experiments/{experiment_namecode}"
competitors = listdir(experiment_path)
for lf in competitors:
    _path = join_path(experiment_path, lf)
    if not isdir(_path):
        competitors.remove(lf)

In [9]:
results_dict = {}
for comp_name in competitors:
    datasets_names = listdir(join_path(
        experiment_path, comp_name
    ))
    for dname in datasets_names:
        experiments_numbers = listdir(join_path(
            experiment_path, comp_name, dname
        ))
        for expno in experiments_numbers:
            trackers_path = join_path(
                experiment_path, comp_name, dname, expno
            )
            evalt_path = join_path(trackers_path, "results.json")
            eval_tracker = load_json_dict(evalt_path)
            results_dict = update_results_dict(comp_name, dname, expno, results_dict, eval_tracker)

In [10]:
summary = summarize_results(results_dict)

In [11]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

def parse_mean_std(value):
    try:
        mean_str, std_str = value.split("±")
        return float(mean_str), float(std_str)
    except:
        return None, None

def save_highlighted_summary(summary_dict, save_path):
    all_rows = []
    metric_names = set()
    excluded_metrics = {"predicted_labels", "ari_labeled", "ami_labeled"}

    # Step 1: Build the rows for the DataFrame
    for dataset_name, alg_results in summary_dict.items():
        for algorithm_name, metrics in alg_results.items():
            if not isinstance(metrics, dict):
                continue
            row = {
                "Dataset": dataset_name,
                "Algorithm": algorithm_name,
            }
            if not isinstance(metrics, dict):
                continue
            for metric_name, value in metrics.items():
                if metric_name in excluded_metrics:
                    continue
                row[metric_name] = value
                metric_names.add(metric_name)
            all_rows.append(row)

    # Step 2: Create full DataFrame
    df = pd.DataFrame(all_rows).fillna("-")

    if df.empty:
        print("⚠️ No data to write.")
        return

    # Step 3: Pivot tables (one for ARI, one for AMI)
    def create_metric_table(metric_name):
        return df.pivot(index="Algorithm", columns="Dataset", values=metric_name)

    tables = {}
    for metric in ["ari_total", "ami_total"]:
        if metric in metric_names:
            tables[metric] = create_metric_table(metric)

    # Step 4: Write tables to Excel
    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        start_row = 1
        table_start_rows = {}
        for metric in ["ari_total", "ami_total"]:
            if metric not in tables:
                continue
            table = tables[metric]
            table.to_excel(writer, sheet_name="Results", startrow=start_row)
            table_start_rows[metric] = start_row
            start_row += len(table) + 5  # space after each table

    # Step 5: Add labels and highlight max values using openpyxl
    wb = load_workbook(save_path)
    ws = wb["Results"]
    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    bold_font = Font(bold=True)

    for metric, start_row in table_start_rows.items():
        label_text = f"{'ARI' if 'ari' in metric else 'AMI'} Scores ({metric})"
        ws.insert_rows(start_row)  # Insert a new row for the label
        ws.cell(row=start_row, column=1, value=label_text).font = bold_font

        table = tables[metric]
        num_rows = table.shape[0]
        num_cols = table.shape[1]

        data_start_row = start_row + 2  # skip label and header row
        for col_index in range(2, 2 + num_cols):  # dataset columns start at column 2
            values = []
            for row_index in range(data_start_row, data_start_row + num_rows):
                cell_value = ws.cell(row=row_index, column=col_index).value
                mean, _ = parse_mean_std(str(cell_value))
                if mean is not None:
                    values.append((mean, row_index))

            if values:
                best_row = max(values, key=lambda x: x[0])[1]
                ws.cell(row=best_row, column=col_index).fill = green_fill

    wb.save(save_path)
    print(f"\n✅ Excel with labeled ARI and AMI tables saved to: {save_path}")


In [12]:
save_highlighted_summary(summary, join_path(experiment_path, f"results_summary_{experiment_namecode}.xlsx"))


✅ Excel with labeled ARI and AMI tables saved to: /export/share/peters57dm/Verbund/deepsync/experiments/competitors203/results_summary_competitors203.xlsx
